# MVP v0.3.1: Multi-Target-Policy OPE with Diffusion Policy Targets

**Date:** 2026-03-10
**Builds on:** MVP v0.3 (failed — BC_Gaussian too weak) + v0.2 (guidance proxy approach)

**Key lesson from v0.3:** BC_Gaussians can't solve Lift (0% success), so they can't be target policies.
But they work fine as guidance proxies (v0.2: 25.9% relative error). So we separate the two roles:

- **Target policy** = robomimic diffusion policy checkpoint (strong enough to solve Lift)
- **Guidance proxy** = BC_Gaussian trained on each target policy's rollouts (only needs `grad_log_prob`)

**Pipeline:**
1. Create HDF5 filter masks for demo subsets of held-out data (demos 150-199)
2. Train 5 robomimic diffusion policies on {50, 40, 30, 20, 10} held-out demos (SLURM)
3. Load SOPE chunk diffusion model (reuse from v0.3, trained on demos 0-149)
4. Oracle rollouts for each trained diffusion policy
5. Collect rollouts from each target policy → train BC_Gaussian guidance proxies
6. Guided SOPE trajectory generation for each target policy
7. Multi-policy evaluation: Spearman ρ, Regret@1

**Data partition (same as v0.3):**
- Demos 0-149 (150): SOPE chunk diffusion model (behavior data / world model)
- Demos 150-199 (50): Train robomimic diffusion policy targets on subsets

In [ ]:
import sys, os
import numpy as np
import torch
import torch.nn as nn
import h5py
import json
import matplotlib.pyplot as plt
from pathlib import Path
from copy import deepcopy
from tqdm import tqdm

# Project root
PROJECT_ROOT = Path("/home1/reishuen/latent_sope")
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "src"))
sys.path.insert(0, str(PROJECT_ROOT / "third_party" / "sope"))
sys.path.insert(0, str(PROJECT_ROOT / "third_party" / "robomimic"))

# SOPE imports
from opelab.core.baselines.diffusion.temporal import TemporalUnet
from opelab.core.baselines.diffusion.diffusion import GaussianDiffusion
from opelab.core.baselines.diffusion.helpers import EMA, apply_conditioning

# Robomimic imports
from latent_sope.robomimic_interface.checkpoints import (
    load_checkpoint, build_env_from_checkpoint, build_rollout_policy_from_checkpoint
)
from latent_sope.eval.oracle import oracle_value
from latent_sope.eval.metrics import multi_policy_ope_eval

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Paths
DEMO_HDF5 = PROJECT_ROOT / "third_party/robomimic/datasets/lift/ph/low_dim_v15.hdf5"
BASE_CKPT_DIR = PROJECT_ROOT / "third_party/robomimic/diffusion_policy_trained_models/test/20260309132349"
SOPE_DIFF_DIR = PROJECT_ROOT / "diffusion_ckpts" / "mvp_v03_diffusion"
TRAINING_OUTPUT_DIR = PROJECT_ROOT / "diffusion_ckpts" / "mvp_v031_target_policies"

## Configuration

In [ ]:
# ── Obs/action dimension config (same as v0.1-v0.3) ──
OBS_KEYS = ["object", "robot0_eef_pos", "robot0_eef_quat", "robot0_gripper_qpos"]
STATE_KEEP_INDICES = [0, 1, 2, 7, 8, 9, 10, 11, 12, 17, 18]  # 11 dims (no quaternions)
ACTION_KEEP_INDICES = [0, 1, 2, 6]  # 4 dims (position + gripper, no orientation)

STATE_DIM = len(STATE_KEEP_INDICES)   # 11
ACTION_DIM = len(ACTION_KEEP_INDICES) # 4
FULL_ACTION_DIM = 7  # robomimic policies output full 7-dim actions
TRANSITION_DIM = STATE_DIM + ACTION_DIM  # 15

# ── Data partition ──
N_BEHAVIOR_DEMOS = 150  # demos 0-149 → SOPE chunk diffusion model
N_TARGET_DEMOS = 50     # demos 150-199 → robomimic diffusion policy targets

# ── SOPE chunk diffusion config (reuse v0.3 model) ──
CHUNK_SIZE = 4
N_DIFFUSION_STEPS = 256
DIM_MULTS = (1, 4, 8)
BASE_DIM = 32
ACTION_WEIGHT = 5.0
PREDICT_EPSILON = False

# ── Multi-policy config ──
TARGET_DEMO_COUNTS = [50, 40, 30, 20, 10]
POLICY_NAMES = [f"diffpol_{n}demos" for n in TARGET_DEMO_COUNTS]

# ── BC_Gaussian guidance proxy config ──
BC_HIDDEN_DIM = 64
BC_TRAIN_EPOCHS = 200
BC_LR = 1e-3
BC_BATCH_SIZE = 256
N_PROXY_ROLLOUTS = 50  # rollouts per policy for BC proxy training

# ── Oracle config ──
N_ORACLE_ROLLOUTS = 20
HORIZON = 60

# ── OPE config ──
GUIDANCE_SCALE = 0.1  # best scale from v0.2
NUM_SYNTHETIC_TRAJS = 50
T_GEN = 60
GAMMA = 1.0
NORMALIZE_GRAD = True

# ── Reward ──
CUBE_Z_INDEX = 2
LIFT_THRESHOLD = 0.84

# ── Robomimic training config ──
ROBOMIMIC_NUM_EPOCHS = 50  # reduced from 2000 for quick pipeline test
ROBOMIMIC_EPOCH_EVERY_N_STEPS = 100  # 100 gradient steps per epoch

print(f"Data partition: {N_BEHAVIOR_DEMOS} behavior demos, {N_TARGET_DEMOS} target demos")
print(f"Target policies: {POLICY_NAMES}")
print(f"Robomimic training: {ROBOMIMIC_NUM_EPOCHS} epochs × {ROBOMIMIC_EPOCH_EVERY_N_STEPS} steps/epoch")
print(f"BC proxy: {N_PROXY_ROLLOUTS} rollouts per policy, {BC_TRAIN_EPOCHS} epochs")
print(f"Oracle: {N_ORACLE_ROLLOUTS} rollouts per policy")
print(f"OPE: {NUM_SYNTHETIC_TRAJS} synthetic trajs, guidance_scale={GUIDANCE_SCALE}")

## Phase 1: Create HDF5 Filter Masks & Training Configs

Create filter masks in the demo HDF5 so robomimic can train on specific demo subsets.
Each mask selects demos from the held-out partition (150-199).

In [ ]:
# ── Step 1a: Create filter masks in HDF5 ──
# robomimic reads mask/<filter_key> from HDF5 to select demo subsets
# Each mask is an array of demo key strings like ["demo_150", "demo_151", ...]

with h5py.File(DEMO_HDF5, "a") as f:
    # Ensure mask group exists
    if "mask" not in f:
        f.create_group("mask")
    mask_group = f["mask"]

    all_demo_keys = sorted(f["data"].keys(), key=lambda x: int(x.split("_")[1]))
    print(f"Total demos in HDF5: {len(all_demo_keys)}")

    for n_demos, name in zip(TARGET_DEMO_COUNTS, POLICY_NAMES):
        mask_name = f"target_{n_demos}"
        # Select demos 150 through 150+n_demos-1
        selected = all_demo_keys[N_BEHAVIOR_DEMOS : N_BEHAVIOR_DEMOS + n_demos]

        # Delete existing mask if present, then create
        if mask_name in mask_group:
            del mask_group[mask_name]
        mask_group.create_dataset(mask_name, data=np.array(selected, dtype="S"))

        print(f"  {mask_name}: {len(selected)} demos "
              f"({selected[0]} → {selected[-1]})")

print("\nFilter masks created.")

In [ ]:
# ── Step 1b: Generate training configs for each target policy ──
# Based on the existing diffusion policy config, modified for each demo subset

base_config = json.loads((BASE_CKPT_DIR / "config.json").read_text())
TRAINING_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Remove keys that exist in saved checkpoint config but not in robomimic's
# default DiffusionPolicyConfig (causes key-lock error during config.update())
for extra_key in ["num_train_batches", "num_epochs"]:
    base_config["algo"]["optim_params"]["policy"].pop(extra_key, None)

config_paths = {}
for n_demos, name in zip(TARGET_DEMO_COUNTS, POLICY_NAMES):
    config = deepcopy(base_config)

    # Point to the full HDF5 (filter mask selects the subset)
    config["train"]["data"] = [{"path": str(DEMO_HDF5)}]
    config["train"]["hdf5_filter_key"] = f"target_{n_demos}"

    # Experiment name and output
    config["experiment"]["name"] = name
    config["train"]["output_dir"] = str(TRAINING_OUTPUT_DIR)
    config["train"]["num_epochs"] = ROBOMIMIC_NUM_EPOCHS

    # Save checkpoints: every 200 epochs + specific milestones
    config["experiment"]["save"]["every_n_epochs"] = 200
    config["experiment"]["save"]["epochs"] = [50, 100, 500, 1000]
    config["experiment"]["save"]["on_best_rollout_success_rate"] = True

    # Disable rollout evaluation during training (saves time, we'll eval separately)
    config["experiment"]["rollout"]["enabled"] = False
    config["experiment"]["render_video"] = False

    # Save config
    config_path = TRAINING_OUTPUT_DIR / f"{name}_config.json"
    config_path.write_text(json.dumps(config, indent=2))
    config_paths[name] = config_path

    print(f"  {name}: filter_key=target_{n_demos}, config saved to {config_path.name}")

print(f"\n{len(config_paths)} training configs saved to {TRAINING_OUTPUT_DIR}")

## Phase 2: Train Robomimic Diffusion Policies

Train 5 diffusion policies on different subsets of held-out demos.
This is the slow step — submit as SLURM jobs or run inline.

**SLURM submission (recommended for overnight):**
```bash
for name in diffpol_50demos diffpol_40demos diffpol_30demos diffpol_20demos diffpol_10demos; do
    sbatch scripts/run_notebook.sbatch \
        --wrap "cd /home1/reishuen/latent_sope/third_party/robomimic && \
        python -m robomimic.scripts.train \
        --config /home1/reishuen/latent_sope/diffusion_ckpts/mvp_v031_target_policies/${name}_config.json"
done
```

**Or run inline below** (will take a long time).

In [ ]:
# ── Train all 5 policies inline ──
# WARNING: This will take a long time. Consider using SLURM instead.
import subprocess

TRAIN_POLICIES = True  # Set to False if policies are already trained

if TRAIN_POLICIES:
    for name in POLICY_NAMES:
        config_path = TRAINING_OUTPUT_DIR / f"{name}_config.json"
        print(f"\n{'='*60}")
        print(f"Training {name}")
        print(f"Config: {config_path}")
        print(f"{'='*60}")

        result = subprocess.run(
            [
                sys.executable, "-m", "robomimic.scripts.train",
                "--config", str(config_path),
            ],
            cwd=str(PROJECT_ROOT / "third_party" / "robomimic"),
            env={
                **os.environ,
                "PYTHONNOUSERSITE": "1",
                "CUDA_VISIBLE_DEVICES": os.environ.get("CUDA_VISIBLE_DEVICES", "0"),
            },
            capture_output=True,
            text=True,
        )

        if result.returncode != 0:
            print(f"FAILED (exit code {result.returncode})")
            print(f"STDERR (last 20 lines):")
            for line in result.stderr.strip().split("\n")[-20:]:
                print(f"  {line}")
        else:
            print(f"SUCCESS")
            # Print last few lines of stdout to show final metrics
            for line in result.stdout.strip().split("\n")[-5:]:
                print(f"  {line}")
else:
    print("Skipping training — TRAIN_POLICIES=False")
    print("Expecting pre-trained policies in:", TRAINING_OUTPUT_DIR)

In [ ]:
# ── Discover trained policy checkpoints ──
# robomimic saves to: output_dir/experiment_name/timestamp/
# Each run dir contains: config.json, models/, logs/

target_policy_ckpts = {}

for name in POLICY_NAMES:
    policy_dir = TRAINING_OUTPUT_DIR / name
    if not policy_dir.exists():
        print(f"  WARNING: {name} — directory not found: {policy_dir}")
        continue

    # Find the most recent timestamp subdirectory
    run_dirs = sorted([d for d in policy_dir.iterdir() if d.is_dir()],
                      key=lambda d: d.name)
    if not run_dirs:
        print(f"  WARNING: {name} — no run directories found in {policy_dir}")
        continue

    run_dir = run_dirs[-1]  # latest timestamp

    # Load checkpoint using our existing infrastructure
    try:
        ckpt = load_checkpoint(str(run_dir))
        target_policy_ckpts[name] = ckpt
        print(f"  {name}: loaded from {run_dir.name}, epoch={ckpt.epoch}")
    except FileNotFoundError as e:
        # Try loading last.pth directly if models/ is empty
        last_pth = run_dir / "last.pth"
        if last_pth.exists():
            ckpt = load_checkpoint(str(run_dir), ckpt_path="last.pth")
            target_policy_ckpts[name] = ckpt
            print(f"  {name}: loaded last.pth from {run_dir.name}")
        else:
            print(f"  WARNING: {name} — {e}")

print(f"\nLoaded {len(target_policy_ckpts)}/{len(POLICY_NAMES)} target policy checkpoints")

## Phase 3: Load Demo Data & SOPE Chunk Diffusion Model

Load the behavior demo data (for normalization stats and initial states)
and the pre-trained SOPE chunk diffusion model from v0.3.

In [ ]:
# ── Load demo data for normalization stats (same as v0.3) ──
def load_robomimic_demos(hdf5_path, obs_keys, state_keep_idx, action_keep_idx):
    """Load robomimic demos from HDF5 → list of {states, actions} dicts."""
    data = []
    with h5py.File(hdf5_path, "r") as f:
        demo_keys = sorted(f["data"].keys(), key=lambda x: int(x.split("_")[1]))
        print(f"Found {len(demo_keys)} demos")
        for dk in tqdm(demo_keys, desc="Loading demos"):
            demo = f["data"][dk]
            obs_arrays = [demo["obs"][k][:] for k in obs_keys]
            full_state = np.concatenate(obs_arrays, axis=-1)
            state = full_state[:, state_keep_idx].astype(np.float32)
            full_actions = demo["actions"][:].astype(np.float32)
            actions = full_actions[:, action_keep_idx]
            rewards = demo["rewards"][:].astype(np.float32)
            episode = {
                "states": state[:-1],
                "actions": actions[:-1],
                "rewards": rewards[:-1],
                "next-states": state[1:],
            }
            data.append(episode)
    total_transitions = sum(len(ep["states"]) for ep in data)
    print(f"Loaded {len(data)} episodes, {total_transitions} total transitions")
    return data

all_data = load_robomimic_demos(DEMO_HDF5, OBS_KEYS, STATE_KEEP_INDICES, ACTION_KEEP_INDICES)
behavior_data = all_data[:N_BEHAVIOR_DEMOS]

# Normalization stats from BEHAVIOR data only
behavior_states_full = np.concatenate(
    [np.concatenate([ep["states"], ep["next-states"][-1:]], axis=0) for ep in behavior_data], axis=0
)
behavior_actions = np.concatenate([ep["actions"] for ep in behavior_data], axis=0)

mean_state = np.mean(behavior_states_full, axis=0)
std_state = np.std(behavior_states_full, axis=0)
mean_action = np.mean(behavior_actions, axis=0)
std_action = np.std(behavior_actions, axis=0)
norm_mean = np.concatenate([mean_state, mean_action]).astype(np.float32)
norm_std = np.maximum(np.concatenate([std_state, std_action]).astype(np.float32), 1e-6)
norm_mean_t = torch.tensor(norm_mean, device=device)
norm_std_t = torch.tensor(norm_std, device=device)
normalize_fn = lambda x: (x - norm_mean_t) / norm_std_t
unnormalize_fn = lambda x: x * norm_std_t + norm_mean_t

print(f"\nBehavior data: {len(behavior_data)} demos")
print(f"Normalization stats: mean shape={norm_mean.shape}, std shape={norm_std.shape}")

In [ ]:
# ── Load pre-trained SOPE chunk diffusion model (from v0.3) ──
temporal_model = TemporalUnet(
    horizon=CHUNK_SIZE,
    transition_dim=TRANSITION_DIM,
    dim=BASE_DIM,
    dim_mults=DIM_MULTS,
    attention=False,
).to(device)

diffusion_model = GaussianDiffusion(
    model=temporal_model,
    horizon=CHUNK_SIZE,
    observation_dim=STATE_DIM,
    action_dim=ACTION_DIM,
    n_timesteps=N_DIFFUSION_STEPS,
    normalizer=normalize_fn,
    unnormalizer=unnormalize_fn,
    predict_epsilon=PREDICT_EPSILON,
    loss_type="l2",
    clip_denoised=False,
    action_weight=ACTION_WEIGHT,
    loss_weights=None,
    loss_discount=1.0,
).to(device)

ema_weights = torch.load(SOPE_DIFF_DIR / "diffusion_model_ema.pt", map_location=device)
diffusion_model.load_state_dict(ema_weights)
diffusion_model.eval()

n_params = sum(p.numel() for p in diffusion_model.parameters())
print(f"Loaded SOPE chunk diffusion model from {SOPE_DIFF_DIR}: {n_params:,} parameters")

## Phase 4: Oracle Rollouts

Roll out each trained diffusion policy in the Lift environment to get ground-truth V^π.
Uses `oracle_value()` from `eval/oracle.py` which handles policy loading and rollouts.

In [ ]:
import robomimic.utils.obs_utils as ObsUtils
import robomimic.utils.file_utils as FileUtils

oracle_results = {}
ORACLE_SAVE_DIR = TRAINING_OUTPUT_DIR / "oracle_rollouts"
ORACLE_SAVE_DIR.mkdir(parents=True, exist_ok=True)

for name in POLICY_NAMES:
    if name not in target_policy_ckpts:
        print(f"Skipping {name} — no checkpoint loaded")
        continue

    print(f"\n{'='*60}")
    print(f"Oracle rollouts for {name}")
    print(f"{'='*60}")

    ckpt = target_policy_ckpts[name]
    result = oracle_value(
        ckpt,
        num_rollouts=N_ORACLE_ROLLOUTS,
        horizon=HORIZON,
        gamma=GAMMA,
        device=str(device),
        verbose=True,
    )

    oracle_results[name] = result
    print(f"  V^π = {result.mean_return:.3f} ± {result.std_return:.3f}")

# Save oracle results
from latent_sope.eval.oracle import save_oracle_result

for name, result in oracle_results.items():
    save_oracle_result(
        ORACLE_SAVE_DIR / f"{name}_oracle.json",
        result,
        policy_name=name,
    )

# Summary table
print(f"\n{'='*60}")
print(f"{'Policy':<20} {'V^π':>8} {'Std':>8} {'Success':>8}")
print(f"{'-'*46}")
for name in POLICY_NAMES:
    if name in oracle_results:
        r = oracle_results[name]
        success = float((r.returns > 0).mean())
        print(f"{name:<20} {r.mean_return:>8.3f} {r.std_return:>8.3f} {success*100:>7.1f}%")

print(f"\nOracle results saved to {ORACLE_SAVE_DIR}")

## Phase 5: Train BC_Gaussian Guidance Proxies

For each target diffusion policy:
1. Collect rollouts to get (state, action) pairs in reduced space
2. Train a BC_Gaussian proxy on those pairs

The BC_Gaussian only needs to approximate `grad_log_prob(a|s)` for guidance — it doesn't need to solve the task.

In [ ]:
class BCGaussian(nn.Module):
    """BC policy with Gaussian action distribution (guidance proxy only)."""

    def __init__(self, state_dim, action_dim, hidden_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, action_dim),
        )
        self.log_std = nn.Parameter(torch.zeros(action_dim))

    def forward(self, state):
        mean = self.net(state)
        std = torch.exp(self.log_std)
        return mean, std

    def log_prob_extended(self, state, action):
        mean, std = self.forward(state)
        log_prob = -0.5 * ((action - mean) / std) ** 2 - self.log_std - 0.5 * np.log(2 * np.pi)
        return log_prob.sum(dim=-1)


def obs_dict_to_reduced_state(obs, obs_keys, state_keep_idx):
    """Convert robomimic obs dict → reduced state vector."""
    full = np.concatenate([obs[k].flatten() for k in obs_keys])
    return full[state_keep_idx].astype(np.float32)


def collect_policy_rollouts(ckpt, obs_keys, state_keep_idx, action_keep_idx,
                            n_rollouts, horizon, device_str):
    """Collect (reduced_state, reduced_action) pairs from a robomimic policy.

    Returns arrays of shape (N_transitions, state_dim) and (N_transitions, action_dim).
    Also saves full rollout data for reuse.
    """
    policy = build_rollout_policy_from_checkpoint(ckpt, device=device_str, verbose=False)
    env = build_env_from_checkpoint(ckpt, render=False, render_offscreen=False, verbose=False)

    all_states = []
    all_actions = []

    for i in range(n_rollouts):
        policy.start_episode()
        obs = env.reset()
        state_dict = env.get_state()
        obs = env.reset_to(state_dict)

        for t in range(horizon):
            state = obs_dict_to_reduced_state(obs, obs_keys, state_keep_idx)
            act_full = policy(ob=obs)  # 7-dim action from diffusion policy
            act_reduced = act_full[action_keep_idx]

            all_states.append(state)
            all_actions.append(act_reduced.astype(np.float32))

            next_obs, reward, done, info = env.step(act_full)
            if done or env.is_success()["task"]:
                break
            obs = deepcopy(next_obs)

        if (i + 1) % max(1, n_rollouts // 5) == 0:
            print(f"    rollout [{i+1}/{n_rollouts}]")

    return np.array(all_states), np.array(all_actions)


def train_bc_gaussian(states, actions, state_dim, action_dim, hidden_dim,
                      epochs, lr, batch_size, device):
    """Train a BCGaussian on (state, action) pairs."""
    policy = BCGaussian(state_dim, action_dim, hidden_dim).to(device)
    optimizer = torch.optim.Adam(policy.parameters(), lr=lr)

    states_t = torch.tensor(states, dtype=torch.float32, device=device)
    actions_t = torch.tensor(actions, dtype=torch.float32, device=device)
    n = len(states_t)

    losses = []
    for epoch in range(epochs):
        idx = torch.randperm(n, device=device)[:batch_size]
        s_batch = states_t[idx]
        a_batch = actions_t[idx]

        log_prob = policy.log_prob_extended(s_batch, a_batch)
        loss = -log_prob.mean()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())

    return policy, losses

print("BC_Gaussian proxy functions defined.")

In [ ]:
# ── Collect rollouts & train BC proxies for each target policy ──
import logging

BC_SAVE_DIR = TRAINING_OUTPUT_DIR / "bc_proxies"
BC_SAVE_DIR.mkdir(parents=True, exist_ok=True)
ROLLOUT_SAVE_DIR = TRAINING_OUTPUT_DIR / "proxy_rollouts"
ROLLOUT_SAVE_DIR.mkdir(parents=True, exist_ok=True)

bc_proxies = {}

# Suppress noisy logging during rollout collection
prev_level = logging.getLogger().level

for name in POLICY_NAMES:
    if name not in target_policy_ckpts:
        print(f"Skipping {name} — no checkpoint")
        continue

    print(f"\n{'='*60}")
    print(f"BC proxy for {name}")
    print(f"{'='*60}")

    ckpt = target_policy_ckpts[name]

    # Step 1: Collect rollouts
    print(f"  Collecting {N_PROXY_ROLLOUTS} rollouts...")
    states, actions = collect_policy_rollouts(
        ckpt, OBS_KEYS, STATE_KEEP_INDICES, ACTION_KEEP_INDICES,
        N_PROXY_ROLLOUTS, HORIZON, str(device),
    )
    print(f"  Collected {len(states)} transitions, "
          f"states {states.shape}, actions {actions.shape}")

    # Save rollout data
    np.savez(
        ROLLOUT_SAVE_DIR / f"{name}_rollouts.npz",
        states=states, actions=actions,
    )

    # Step 2: Train BC_Gaussian
    print(f"  Training BC_Gaussian ({BC_TRAIN_EPOCHS} epochs)...")
    bc_policy, bc_losses = train_bc_gaussian(
        states, actions, STATE_DIM, ACTION_DIM, BC_HIDDEN_DIM,
        BC_TRAIN_EPOCHS, BC_LR, BC_BATCH_SIZE, device,
    )
    bc_policy.eval()
    bc_proxies[name] = bc_policy

    # Save BC checkpoint
    torch.save({
        "state_dict": bc_policy.state_dict(),
        "state_dim": STATE_DIM,
        "action_dim": ACTION_DIM,
        "hidden_dim": BC_HIDDEN_DIM,
        "n_transitions": len(states),
        "train_epochs": BC_TRAIN_EPOCHS,
        "final_loss": bc_losses[-1],
    }, BC_SAVE_DIR / f"{name}_bc_proxy.pt")

    learned_std = torch.exp(bc_policy.log_std).detach().cpu().numpy()
    print(f"  Final NLL: {bc_losses[-1]:.3f}")
    print(f"  Learned std: {learned_std}")

print(f"\n{len(bc_proxies)} BC proxies trained and saved to {BC_SAVE_DIR}")
print(f"Rollout data saved to {ROLLOUT_SAVE_DIR}")

## Phase 6: Guided SOPE Trajectory Generation

Generate synthetic trajectories for each target policy using BC_Gaussian guidance proxies.
Initial states sampled from the behavior data (what the SOPE diffusion model was trained on).

In [ ]:
def get_initial_states_from_data(offline_data, num_samples, device):
    """Sample initial states from demo dataset."""
    all_initial = np.array([ep["states"][0] for ep in offline_data])
    indices = np.random.choice(len(all_initial), num_samples, replace=True)
    return torch.tensor(all_initial[indices], dtype=torch.float32, device=device)


def generate_guided_trajectories(
    diffusion_model, policy, initial_states,
    normalize_fn, unnormalize_fn,
    state_dim, action_dim, chunk_size, t_gen,
    action_scale, normalize_grad, device,
):
    """Generate full trajectories via guided autoregressive stitching.

    Same implementation as v0.3 — guidance via grad_a log pi(a|s) from BC proxy.
    """
    batch_size = initial_states.shape[0]
    transition_dim = state_dim + action_dim

    # Normalize initial states for conditioning
    padded = torch.cat([
        initial_states,
        torch.zeros(batch_size, action_dim, device=device)
    ], dim=1)
    normalized_initial = normalize_fn(padded)[:, :state_dim]

    all_trajectories = torch.zeros(batch_size, t_gen, transition_dim, device=device)
    conditions = {0: normalized_initial}
    total_generated = 0

    while total_generated < t_gen:
        steps_remaining = t_gen - total_generated
        shape = (batch_size, chunk_size, transition_dim)

        x = torch.randn(shape, device=device)
        x = apply_conditioning(x, conditions, state_dim)

        for t_diff in reversed(range(diffusion_model.n_timesteps)):
            t_tensor = torch.full((batch_size,), t_diff, device=device, dtype=torch.long)
            model_mean, _, model_log_variance = diffusion_model.p_mean_variance(x=x, t=t_tensor)
            model_std = torch.exp(0.5 * model_log_variance)

            # ── Guidance step ──
            if action_scale > 0 and policy is not None:
                mean_unnorm = unnormalize_fn(model_mean)
                states_flat = mean_unnorm[:, :, :state_dim].reshape(-1, state_dim).detach()
                actions_flat = mean_unnorm[:, :, state_dim:].reshape(-1, action_dim).detach().requires_grad_(True)

                log_prob = policy.log_prob_extended(states_flat, actions_flat)
                total_log_prob = log_prob.sum()
                grad_action = torch.autograd.grad(total_log_prob, actions_flat)[0]

                if normalize_grad:
                    grad_norm = grad_action.norm(dim=-1, keepdim=True) + 1e-6
                    grad_action = grad_action / grad_norm

                guide = torch.zeros_like(mean_unnorm)
                guide[:, :, state_dim:] = grad_action.reshape(batch_size, chunk_size, action_dim)

                mean_unnorm = mean_unnorm + action_scale * guide
                model_mean = normalize_fn(mean_unnorm)
                model_mean = apply_conditioning(model_mean, conditions, state_dim)

            noise = torch.randn_like(x)
            nonzero_mask = float(t_diff != 0)
            x = model_mean + nonzero_mask * model_std * noise
            x = apply_conditioning(x, conditions, state_dim)

        chunk_unnorm = unnormalize_fn(x)
        steps_to_store = min(chunk_size - 1, steps_remaining)
        all_trajectories[:, total_generated:total_generated + steps_to_store] = chunk_unnorm[:, :steps_to_store]
        total_generated += steps_to_store

        if total_generated >= t_gen:
            break
        conditions = {0: x[:, -1, :state_dim]}

    return all_trajectories.detach().cpu().numpy()


def score_trajectories_gt(trajectories, cube_z_index, threshold, gamma=1.0):
    """Score with ground-truth Lift reward: cube_z > threshold → 1.0 per step."""
    B, T, D = trajectories.shape
    returns = np.zeros(B)
    for i in range(B):
        gamma_t = 1.0
        for t in range(T):
            if trajectories[i, t, cube_z_index] > threshold:
                returns[i] += gamma_t
            gamma_t *= gamma
    return returns

print("Guided generation and scoring functions defined.")

In [ ]:
# ── Run guided SOPE for each target policy ──
ope_results = {}

# Use same initial states for all policies (fair comparison)
torch.manual_seed(123)
initial_states = get_initial_states_from_data(behavior_data, NUM_SYNTHETIC_TRAJS, device)

for name in POLICY_NAMES:
    if name not in bc_proxies:
        print(f"Skipping {name} — no BC proxy")
        continue

    print(f"\n{'='*60}")
    print(f"Guided SOPE for {name} (scale={GUIDANCE_SCALE})")
    print(f"{'='*60}")

    synthetic_trajs = generate_guided_trajectories(
        diffusion_model=diffusion_model,
        policy=bc_proxies[name],
        initial_states=initial_states,
        normalize_fn=normalize_fn,
        unnormalize_fn=unnormalize_fn,
        state_dim=STATE_DIM,
        action_dim=ACTION_DIM,
        chunk_size=CHUNK_SIZE,
        t_gen=T_GEN,
        action_scale=GUIDANCE_SCALE,
        normalize_grad=NORMALIZE_GRAD,
        device=device,
    )

    returns = score_trajectories_gt(synthetic_trajs, CUBE_Z_INDEX, LIFT_THRESHOLD, GAMMA)

    ope_results[name] = {
        "returns": returns,
        "trajectories": synthetic_trajs,
        "estimate": float(returns.mean()),
        "std": float(returns.std()),
        "success_rate": float((returns > 0).mean()),
    }

    oracle_v = oracle_results[name].mean_return
    ope_v = returns.mean()
    rel_err = abs(ope_v - oracle_v) / max(abs(oracle_v), 1e-6)
    print(f"  Oracle V^π = {oracle_v:.3f}")
    print(f"  OPE estimate = {ope_v:.3f} ± {returns.std():.3f}")
    print(f"  Relative error = {rel_err:.2%}")
    print(f"  Synthetic success rate = {(returns > 0).mean()*100:.1f}%")

# Save OPE results
OPE_SAVE_DIR = TRAINING_OUTPUT_DIR / "ope_results"
OPE_SAVE_DIR.mkdir(parents=True, exist_ok=True)
for name in ope_results:
    np.savez(
        OPE_SAVE_DIR / f"{name}_ope.npz",
        returns=ope_results[name]["returns"],
        trajectories=ope_results[name]["trajectories"],
    )
print(f"\nOPE results saved to {OPE_SAVE_DIR}")
print("All policies evaluated.")

## Phase 7: Multi-Policy Evaluation

Compute ranking metrics: Spearman ρ (rank correlation) and Regret@1 (policy selection).

In [ ]:
# Collect oracle and OPE values (only for policies that have both)
eval_names = [n for n in POLICY_NAMES if n in oracle_results and n in ope_results]
oracle_values = [oracle_results[n].mean_return for n in eval_names]
ope_estimates = [ope_results[n]["estimate"] for n in eval_names]
synthetic_returns_list = [ope_results[n]["returns"] for n in eval_names]

mp_result = multi_policy_ope_eval(
    oracle_values=oracle_values,
    ope_estimates=ope_estimates,
    synthetic_returns_per_policy=synthetic_returns_list,
    k_values=(1, 2, 3),
)

print("=" * 60)
print("MULTI-POLICY OPE EVALUATION")
print("=" * 60)

print(f"\n{'Policy':<20} {'Oracle V^π':>12} {'OPE Est':>12} {'Rel Error':>12} {'OPE Std':>10}")
print("-" * 68)
for i, name in enumerate(eval_names):
    pp = mp_result.per_policy[i]
    print(f"{name:<20} {oracle_values[i]:>12.3f} {ope_estimates[i]:>12.3f} "
          f"{pp.relative_error:>11.1%} {pp.ope_std:>10.3f}")

print(f"\nAggregate Metrics:")
print(f"  Mean MSE:            {mp_result.mean_mse:.4f}")
print(f"  Mean Relative Error: {mp_result.mean_relative_error:.2%}")
print(f"  Spearman ρ:          {mp_result.spearman_rho:.4f} (p={mp_result.spearman_pvalue:.4f})")
print(f"  Regret@1:            {mp_result.regret_at_1:.4f}")
for k, reg in mp_result.regret_at_k.items():
    print(f"  Regret@{k}:            {reg:.4f}")

# Rankings side by side
oracle_ranking = np.argsort(oracle_values)[::-1]
ope_ranking = np.argsort(ope_estimates)[::-1]

print(f"\n{'Rank':<6} {'Oracle Best → Worst':<25} {'OPE Best → Worst':<25}")
print("-" * 56)
for rank in range(len(eval_names)):
    print(f"{rank+1:<6} {eval_names[oracle_ranking[rank]]:<25} {eval_names[ope_ranking[rank]]:<25}")

## Phase 8: Visualization

In [ ]:
if len(eval_names) == 0:
    print("No policies to visualize — skipping.")
else:
    fig, axes = plt.subplots(2, 2, figsize=(14, 11))

    # ── Panel 1: Oracle vs OPE scatter plot ──
    ax = axes[0, 0]
    ax.scatter(oracle_values, ope_estimates, s=100, c="steelblue", edgecolors="black", zorder=5)
    for i, name in enumerate(eval_names):
        ax.annotate(name, (oracle_values[i], ope_estimates[i]),
                    textcoords="offset points", xytext=(8, 5), fontsize=8)
    lims = [min(min(oracle_values), min(ope_estimates)) - 0.5,
            max(max(oracle_values), max(ope_estimates)) + 0.5]
    ax.plot(lims, lims, "k--", alpha=0.3, label="y=x (perfect)")
    ax.set_xlabel("Oracle V^π")
    ax.set_ylabel("OPE Estimate")
    ax.set_title(f"Oracle vs OPE (Spearman ρ={mp_result.spearman_rho:.3f})")
    ax.legend()
    ax.grid(True, alpha=0.3)

    # ── Panel 2: Bar chart comparing oracle and OPE per policy ──
    ax = axes[0, 1]
    x_pos = np.arange(len(eval_names))
    width = 0.35
    ax.bar(x_pos - width/2, oracle_values, width, label="Oracle V^π",
           color="steelblue", edgecolor="black")
    ax.bar(x_pos + width/2, ope_estimates, width, label="OPE Estimate",
           color="coral", edgecolor="black")
    ope_stds = [ope_results[n]["std"] for n in eval_names]
    ax.errorbar(x_pos + width/2, ope_estimates, yerr=ope_stds,
                fmt="none", color="black", capsize=3)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(eval_names, rotation=30, ha="right", fontsize=8)
    ax.set_ylabel("Value")
    ax.set_title("Per-Policy: Oracle vs OPE")
    ax.legend()
    ax.grid(True, alpha=0.3, axis="y")

    # ── Panel 3: Ranking comparison ──
    ax = axes[1, 0]
    oracle_ranks = np.argsort(np.argsort(oracle_values)[::-1]) + 1
    ope_ranks = np.argsort(np.argsort(ope_estimates)[::-1]) + 1
    ax.scatter(oracle_ranks, ope_ranks, s=100, c="steelblue", edgecolors="black", zorder=5)
    for i, name in enumerate(eval_names):
        ax.annotate(name, (oracle_ranks[i], ope_ranks[i]),
                    textcoords="offset points", xytext=(8, 5), fontsize=8)
    ax.plot([0.5, len(eval_names)+0.5], [0.5, len(eval_names)+0.5],
            "k--", alpha=0.3, label="Perfect ranking")
    ax.set_xlabel("Oracle Rank (1=best)")
    ax.set_ylabel("OPE Rank (1=best)")
    ax.set_title("Rank Agreement")
    ax.set_xticks(range(1, len(eval_names)+1))
    ax.set_yticks(range(1, len(eval_names)+1))
    ax.legend()
    ax.grid(True, alpha=0.3)

    # ── Panel 4: Relative error per policy ──
    ax = axes[1, 1]
    rel_errors = [mp_result.per_policy[i].relative_error * 100 for i in range(len(eval_names))]
    colors = ["green" if e < 50 else "orange" if e < 100 else "red" for e in rel_errors]
    ax.bar(eval_names, rel_errors, color=colors, edgecolor="black")
    ax.set_ylabel("Relative Error (%)")
    ax.set_title("Per-Policy Relative Error")
    ax.tick_params(axis="x", rotation=30)
    ax.grid(True, alpha=0.3, axis="y")

    plt.suptitle(f"MVP v0.3.1: Multi-Policy OPE (Diffusion Targets) — "
                 f"Spearman ρ={mp_result.spearman_rho:.3f}, "
                 f"Regret@1={mp_result.regret_at_1:.3f}", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Per-policy trajectory visualization: cube_z for each policy ──
fig, axes = plt.subplots(1, len(eval_names), figsize=(4 * len(eval_names), 4), sharey=True)
if len(eval_names) == 1:
    axes = [axes]

for idx, name in enumerate(eval_names):
    ax = axes[idx]
    trajs = ope_results[name]["trajectories"]
    for j in range(min(10, NUM_SYNTHETIC_TRAJS)):
        ax.plot(trajs[j, :, CUBE_Z_INDEX], alpha=0.3, color="coral")
    ax.axhline(y=LIFT_THRESHOLD, color="red", linestyle="--", alpha=0.5)
    oracle_v = oracle_results[name].mean_return
    ope_v = ope_results[name]["estimate"]
    ax.set_title(f"{name}\nV^π={oracle_v:.1f}, OPE={ope_v:.1f}", fontsize=9)
    ax.set_xlabel("Timestep")
    if idx == 0:
        ax.set_ylabel("Cube z-position")
    ax.grid(True, alpha=0.3)

plt.suptitle("Synthetic Trajectories (cube_z) per Policy", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

## Summary

In [ ]:
print("=" * 70)
print("MVP v0.3.1: MULTI-TARGET-POLICY OPE (DIFFUSION TARGETS) — FINAL SUMMARY")
print("=" * 70)

print(f"\nSetup:")
print(f"  Data partition: {N_BEHAVIOR_DEMOS} behavior demos / {N_TARGET_DEMOS} target demos")
print(f"  SOPE diffusion model: reused from v0.3 (trained on behavior split)")
print(f"  Target policies: {len(eval_names)} robomimic diffusion policies on held-out demos {TARGET_DEMO_COUNTS}")
print(f"  Guidance proxies: BC_Gaussian per policy ({N_PROXY_ROLLOUTS} rollouts, {BC_TRAIN_EPOCHS} epochs)")
print(f"  Guidance scale: {GUIDANCE_SCALE}")
print(f"  Oracle rollouts per policy: {N_ORACLE_ROLLOUTS}")
print(f"  Synthetic trajectories per policy: {NUM_SYNTHETIC_TRAJS}")

print(f"\nPer-Policy Results:")
print(f"  {'Policy':<20} {'Demos':>6} {'Oracle':>8} {'OPE':>8} {'RelErr':>8} {'Succ(O)':>8} {'Succ(S)':>8}")
print(f"  {'-'*68}")
for i, name in enumerate(eval_names):
    n_demos = TARGET_DEMO_COUNTS[POLICY_NAMES.index(name)]
    o = oracle_results[name]
    e = ope_results[name]
    pp = mp_result.per_policy[i]
    o_success = float((o.returns > 0).mean())
    print(f"  {name:<20} {n_demos:>5}  {o.mean_return:>8.2f} {e['estimate']:>8.2f} "
          f"{pp.relative_error:>7.1%} {o_success*100:>7.1f}% {e['success_rate']*100:>7.1f}%")

print(f"\nRanking Metrics:")
print(f"  Spearman ρ = {mp_result.spearman_rho:.4f} (p = {mp_result.spearman_pvalue:.4f})")
print(f"  Regret@1   = {mp_result.regret_at_1:.4f}")
for k, reg in mp_result.regret_at_k.items():
    if k > 1:
        print(f"  Regret@{k}   = {reg:.4f}")

print(f"\nAggregate:")
print(f"  Mean MSE            = {mp_result.mean_mse:.4f}")
print(f"  Mean Relative Error = {mp_result.mean_relative_error:.2%}")

print(f"\nSaved Artifacts:")
print(f"  Target policy checkpoints: {TRAINING_OUTPUT_DIR}/")
print(f"  Oracle results:            {ORACLE_SAVE_DIR}/")
print(f"  BC proxy checkpoints:      {BC_SAVE_DIR}/")
print(f"  Proxy rollout data:        {ROLLOUT_SAVE_DIR}/")
print(f"  OPE results:               {OPE_SAVE_DIR}/")

print(f"\n{'='*70}")
print("MVP v0.3.1 COMPLETE")
print("=" * 70)